In [11]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.impute import SimpleImputer
import matplotlib.pyplot as plt

In [12]:
X_feat_12    = np.load('X_feat_12.npy')
Y_71         = np.load('Y_71.npy')
ENFERMEDADES_71 = np.load('ENFERMEDADES_71.npy', allow_pickle=True).tolist()

# Eliminamos NaN
imputer   = SimpleImputer(strategy='mean')
X_feat_12 = imputer.fit_transform(X_feat_12)

print(f"Shape X_feat_12:  {X_feat_12.shape}")
print(f"Shape Y_71:       {Y_71.shape}")
print(f"Enfermedades:     {len(ENFERMEDADES_71)}")

Shape X_feat_12:  (16984, 458)
Shape Y_71:       (16984, 71)
Enfermedades:     71


In [13]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X_feat_12, Y_71, test_size=0.2, random_state=42
)

scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print(f"Train: {X_train.shape} | Test: {X_test.shape}")

Train: (13587, 458) | Test: (3397, 458)


In [14]:
def mostrar_metricas(nombre, Y_true, Y_pred):
    mse_global = mean_squared_error(Y_true, Y_pred)
    mae_global = mean_absolute_error(Y_true, Y_pred)
    print(f"\n--- {nombre} ---")
    print(f"MAE global: {mae_global:.4f} | MSE global: {mse_global:.4f}")
    print(f"\n{'Enfermedad':<12} {'MAE':>6} {'MSE':>6} {'R2':>6} {'Pacientes':>10}")
    print("-" * 50)
    for i, enf in enumerate(ENFERMEDADES_71):
        y_true_i    = Y_true[:, i]
        y_pred_i    = Y_pred[:, i]
        n_positivos = (y_true_i > 0).sum()
        if n_positivos < 5:
            continue
        mae_i = mean_absolute_error(y_true_i, y_pred_i)
        mse_i = mean_squared_error(y_true_i, y_pred_i)
        r2_i  = r2_score(y_true_i, y_pred_i)
        print(f"{enf:<12} {mae_i:>6.3f} {mse_i:>6.3f} {r2_i:>6.3f} {n_positivos:>10}")

def mostrar_prediccion_paciente(idx, Y_true, Y_pred, umbral=0.3):
    print(f"\nPaciente {idx}:")
    print(f"{'Enfermedad':<12} {'Real':>6} {'Predicho':>10}")
    print("-" * 32)
    for i, enf in enumerate(ENFERMEDADES_71):
        real = Y_true[idx, i]
        pred = Y_pred[idx, i]
        if real > 0 or pred > umbral:
            print(f"{enf:<12} {real:>6.2f} {pred:>10.2f}")


def ejecutar_ridge(X_train, X_test, Y_train, Y_test, nombre):
    print(f"\n=== Ridge Regression — {nombre} ===")
    model = MultiOutputRegressor(Ridge(alpha=1.0), n_jobs=-1)
    model.fit(X_train, Y_train)

    Y_pred_train = np.clip(model.predict(X_train), 0, 1)
    Y_pred_test  = np.clip(model.predict(X_test),  0, 1)

    mostrar_metricas("Train", Y_train, Y_pred_train)
    mostrar_metricas("Test",  Y_test,  Y_pred_test)
    return model, Y_pred_test


def ejecutar_random_forest_reg(X_train, X_test, Y_train, Y_test, nombre):
    print(f"\n=== Random Forest Regressor — {nombre} ===")
    model = RandomForestRegressor(
        n_estimators=200, max_depth=20, min_samples_leaf=5,
        random_state=42, n_jobs=-1
    )
    model.fit(X_train, Y_train)

    Y_pred_train = np.clip(model.predict(X_train), 0, 1)
    Y_pred_test  = np.clip(model.predict(X_test),  0, 1)

    mostrar_metricas("Train", Y_train, Y_pred_train)
    mostrar_metricas("Test",  Y_test,  Y_pred_test)
    return model, Y_pred_test


def ejecutar_mlp_reg(X_train, X_test, Y_train, Y_test, nombre):
    print(f"\n=== MLP Regressor — {nombre} ===")

    X_train_t = torch.tensor(X_train, dtype=torch.float32)
    X_test_t  = torch.tensor(X_test,  dtype=torch.float32)
    Y_train_t = torch.tensor(Y_train, dtype=torch.float32)

    train_loader      = DataLoader(TensorDataset(X_train_t, Y_train_t), batch_size=32, shuffle=True)
    train_eval_loader = DataLoader(TensorDataset(X_train_t, Y_train_t), batch_size=32, shuffle=False)
    test_loader       = DataLoader(TensorDataset(X_test_t,  torch.zeros(len(X_test_t))), batch_size=32, shuffle=False)

    class MLPRegressor(nn.Module):
        def __init__(self, input_size, output_size):
            super().__init__()
            self.red = nn.Sequential(
                nn.Linear(input_size, 256),
                nn.BatchNorm1d(256),
                nn.ReLU(),
                nn.Dropout(0.3),
                nn.Linear(256, 128),
                nn.BatchNorm1d(128),
                nn.ReLU(),
                nn.Dropout(0.3),
                nn.Linear(128, 64),
                nn.ReLU(),
                nn.Dropout(0.2),
                nn.Linear(64, output_size),
                nn.Sigmoid()
            )
        def forward(self, x):
            return self.red(x)

    model     = MLPRegressor(input_size=X_train.shape[1], output_size=Y_train.shape[1])
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

    EPOCHS    = 100
    PACIENCIA = 15
    mejor_perdida     = float('inf')
    epochs_sin_mejora = 0
    mejor_estado      = None

    for epoch in range(EPOCHS):
        model.train()
        perdida_total = 0
        for X_batch, Y_batch in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(X_batch), Y_batch)
            loss.backward()
            optimizer.step()
            perdida_total += loss.item()

        perdida_media = perdida_total / len(train_loader)
        scheduler.step(perdida_media)

        if (epoch + 1) % 10 == 0:
            print(f"  Epoch {epoch+1}/{EPOCHS} - Pérdida: {perdida_media:.4f} - lr: {optimizer.param_groups[0]['lr']:.6f}")

        if perdida_media < mejor_perdida - 0.0001:
            mejor_perdida     = perdida_media
            epochs_sin_mejora = 0
            mejor_estado      = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            epochs_sin_mejora += 1
            if epochs_sin_mejora >= PACIENCIA:
                print(f"  Early stopping en epoch {epoch+1}")
                break

    model.load_state_dict(mejor_estado)
    model.eval()

    def predecir(loader):
        preds = []
        with torch.no_grad():
            for X_batch, _ in loader:
                preds.append(model(X_batch).numpy())
        return np.clip(np.vstack(preds), 0, 1)

    Y_pred_train = predecir(train_eval_loader)
    Y_pred_test  = predecir(test_loader)

    mostrar_metricas("Train", Y_train, Y_pred_train)
    mostrar_metricas("Test",  Y_test,  Y_pred_test)
    return model, Y_pred_test

In [15]:
ridge_model, ridge_preds = ejecutar_ridge(X_train, X_test, Y_train, Y_test, "Features 12 deriv.")
rf_model,    rf_preds    = ejecutar_random_forest_reg(X_train, X_test, Y_train, Y_test, "Features 12 deriv.")
mlp_model,   mlp_preds   = ejecutar_mlp_reg(X_train, X_test, Y_train, Y_test, "Features 12 deriv.")


=== Ridge Regression — Features 12 deriv. ===

--- Train ---
MAE global: 0.0246 | MSE global: 0.0092

Enfermedad      MAE    MSE     R2  Pacientes
--------------------------------------------------
NST_          0.054  0.024  0.073        375
ANEUR         0.007  0.001  0.084         61
ISCAS         0.017  0.006  0.059        101
INJLA         0.002  0.000  0.065          7
INJAS         0.023  0.008  0.139        134
2AVB          0.002  0.000  0.078          6
PAC           0.006  0.002  0.065         26
LAFB          0.075  0.028  0.599       1048
RVH           0.011  0.003  0.197         86
RAO/RAE       0.014  0.005  0.084         83
LMI           0.011  0.003  0.141        115
ALMI          0.025  0.007  0.245        183
IMI           0.107  0.039  0.285       1719
LAO/LAE       0.038  0.015  0.094        296
AFLT          0.010  0.003  0.147         43
INJIL         0.002  0.000  0.044          7
IPMI          0.005  0.001  0.085         19
WPW           0.008  0.002  0.129   

In [ ]:
print("=== Predicciones MLP ===")
for idx in [3, 4, 5]:
    mostrar_prediccion_paciente(idx, Y_test, mlp_preds)

=== Predicciones MLP ===

Paciente 3:
Enfermedad     Real   Predicho
--------------------------------
IMI            1.00       0.81
LAO/LAE        1.00       0.01
ISCLA          1.00       0.04
AMI            1.00       0.02

Paciente 4:
Enfermedad     Real   Predicho
--------------------------------
ISC_           1.00       0.78
LVH            1.00       0.86

Paciente 5:
Enfermedad     Real   Predicho
--------------------------------
NORM           0.80       0.72


In [19]:
print("=== Predicciones Ridge Regressor ===")
for idx in [3,4,5]:
    mostrar_prediccion_paciente(idx, Y_test, ridge_preds)

=== Predicciones Ridge Regressor ===

Paciente 3:
Enfermedad     Real   Predicho
--------------------------------
IMI            1.00       0.32
LAO/LAE        1.00       0.07
ISCLA          1.00       0.03
AMI            1.00       0.05

Paciente 4:
Enfermedad     Real   Predicho
--------------------------------
ISC_           1.00       0.27
LVH            1.00       0.51

Paciente 5:
Enfermedad     Real   Predicho
--------------------------------
NORM           0.80       0.57


In [21]:
print("=== Predicciones Random Forest ===")
for idx in [3,4,5]:
    mostrar_prediccion_paciente(idx, Y_test, rf_preds)

=== Predicciones Random Forest ===

Paciente 3:
Enfermedad     Real   Predicho
--------------------------------
IMI            1.00       0.44
LAO/LAE        1.00       0.02
ISCLA          1.00       0.07
AMI            1.00       0.02

Paciente 4:
Enfermedad     Real   Predicho
--------------------------------
ISC_           1.00       0.49
LVH            1.00       0.60

Paciente 5:
Enfermedad     Real   Predicho
--------------------------------
NORM           0.80       0.69


In [22]:
def mostrar_paciente_completo(idx, Y_test, Y_pred, modelo_nombre):
    print(f"\n{'='*50}")
    print(f"Paciente {idx} — Predicción {modelo_nombre}")
    print(f"{'='*50}")
    print(f"\n{'Enfermedad':<12} {'Probabilidad real':>18} {'Probabilidad pred':>18} {'Diferencia':>12}")
    print("-" * 65)
    
    for i, enf in enumerate(ENFERMEDADES_71):
        real = Y_test[idx, i]
        pred = Y_pred[idx, i]
        if real > 0.01 or pred > 0.2:  # mostramos las relevantes
            diff = pred - real
            signo = "+" if diff > 0 else ""
            print(f"{enf:<12} {real:>18.2f} {pred:>18.2f} {signo}{diff:>11.2f}")

# Probamos con tres pacientes distintos
for idx in [0, 5, 10]:
    mostrar_paciente_completo(idx, Y_test, mlp_preds, "MLP")


Paciente 0 — Predicción MLP

Enfermedad    Probabilidad real  Probabilidad pred   Diferencia
-----------------------------------------------------------------
NORM                       1.00               0.91       -0.09

Paciente 5 — Predicción MLP

Enfermedad    Probabilidad real  Probabilidad pred   Diferencia
-----------------------------------------------------------------
NORM                       0.80               0.72       -0.08

Paciente 10 — Predicción MLP

Enfermedad    Probabilidad real  Probabilidad pred   Diferencia
-----------------------------------------------------------------
PACE                       1.00               1.00       -0.00


In [27]:
import random

def diagnosticar_paciente_nuevo(idx, X_test, Y_test, model, ENFERMEDADES_71, umbral=0.3):
    print(f"\n{'='*55}")
    print(f"  INFORME DE DIAGNÓSTICO — PACIENTE #{idx}")
    print(f"{'='*55}")
    
    # Preparamos la entrada
    x = torch.tensor(X_test[idx:idx+1], dtype=torch.float32)
    
    # Predicción
    model.eval()
    with torch.no_grad():
        pred = model(x).numpy()[0]
    
    # Separamos en detectadas y no detectadas
    detectadas    = [(ENFERMEDADES_71[i], pred[i]) for i in range(len(ENFERMEDADES_71)) if pred[i] >= umbral]
    no_detectadas = [(ENFERMEDADES_71[i], pred[i]) for i in range(len(ENFERMEDADES_71)) if pred[i] < umbral and pred[i] > 0.05]
    
    # Ordenamos por probabilidad descendente
    detectadas    = sorted(detectadas,    key=lambda x: -x[1])
    no_detectadas = sorted(no_detectadas, key=lambda x: -x[1])
    
    # Enfermedades detectadas
    if detectadas:
        print(f"\n  Enfermedades detectadas (prob >= {umbral}):")
        print(f"  {'Enfermedad':<12} {'Probabilidad':>14} {'Barra'}")
        print(f"  {'-'*45}")
        for enf, prob in detectadas:
            barra = '█' * int(prob * 20)
            print(f"  {enf:<12} {prob:>13.1%}  {barra}")
    else:
        print(f"\n  No se detectaron enfermedades con probabilidad >= {umbral}")

    # Posibles pero bajas
    if no_detectadas:
        print(f"\n  Señales débiles (prob entre 5% y {int(umbral*100)}%):")
        print(f"  {'Enfermedad':<12} {'Probabilidad':>14}")
        print(f"  {'-'*30}")
        for enf, prob in no_detectadas[:5]:  # mostramos solo las 5 más altas
            print(f"  {enf:<12} {prob:>13.1%}")

    # Diagnóstico real (para validar)
    real = [(ENFERMEDADES_71[i], Y_test[idx, i]) for i in range(len(ENFERMEDADES_71)) if Y_test[idx, i] > 0]
    real = sorted(real, key=lambda x: -x[1])
    print(f"\n  {'─'*45}")
    print(f"  Diagnóstico real (oculto al modelo):")
    for enf, prob in real:
        print(f"  {enf:<12} {prob:>13.1%}")
    print(f"{'='*55}\n")


# Elegimos 3 pacientes interesantes del test
# Buscamos pacientes con varias enfermedades para que sea más ilustrativo
pacientes_interesantes = []
for idx in range(len(Y_test)):
    n_enfermedades = (Y_test[idx] > 0.5).sum()
    if n_enfermedades >= 2:
        pacientes_interesantes.append(idx)

# Mostramos 3 pacientes aleatorios con varias enfermedades
random.seed(4)
seleccionados = random.sample(pacientes_interesantes, 3)

for idx in seleccionados:
    diagnosticar_paciente_nuevo(idx, X_test, Y_test, mlp_model, ENFERMEDADES_71, umbral=0.3)
    


  INFORME DE DIAGNÓSTICO — PACIENTE #1108

  Enfermedades detectadas (prob >= 0.3):
  Enfermedad     Probabilidad Barra
  ---------------------------------------------
  LAFB                 48.0%  █████████
  NORM                 38.2%  ███████

  ─────────────────────────────────────────────
  Diagnóstico real (oculto al modelo):
  LAFB                100.0%
  NORM                100.0%


  INFORME DE DIAGNÓSTICO — PACIENTE #1411

  Enfermedades detectadas (prob >= 0.3):
  Enfermedad     Probabilidad Barra
  ---------------------------------------------
  ISC_                 97.8%  ███████████████████
  LVH                  97.6%  ███████████████████

  Señales débiles (prob entre 5% y 30%):
  Enfermedad     Probabilidad
  ------------------------------
  IMI                  11.0%

  ─────────────────────────────────────────────
  Diagnóstico real (oculto al modelo):
  ISC_                100.0%
  LVH                 100.0%


  INFORME DE DIAGNÓSTICO — PACIENTE #495

  Enfermedade

In [28]:
for idx in seleccionados:
    diagnosticar_paciente_nuevo(idx, X_test, Y_test,rf_model, ENFERMEDADES_71, umbral=0.3)


  INFORME DE DIAGNÓSTICO — PACIENTE #1108


AttributeError: 'RandomForestRegressor' object has no attribute 'eval'

In [ ]:
for idx in seleccionados:
    diagnosticar_paciente_nuevo(idx, X_test, Y_test, ridge_model, ENFERMEDADES_71, umbral=0.3)